In [1]:
## import libraries
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

In [22]:
df = pd.read_csv("../data/processed/amazon_sales_without_outliers.csv")

C:\Users\SVS\AppData\Local\Temp\ipykernel_10352\1940308431.py:1: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/amazon_sales_without_outliers.csv")


In [23]:
## verify dataset 
print(df.shape)
df.head()

(128029, 25)


,idx,order_id,order_date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,currency,amount,ship_city,ship_state,ship_postal_code,ship_country,promotion_ids,b2b,fulfilled_by,unnamed_22,z_score
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,NaN,0,INR,647.62,MUMBAI,Maharashtra,400081.0,IN,NaN,False,Easy Ship,NaN,0.006200
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,Karnataka,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN,-0.879576
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,Maharashtra,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN,-1.161857
3,3,403-9615377-8133951,04-30-22,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,NaN,0,INR,753.33,PUDUCHERRY,Puducherry,605008.0,IN,NaN,False,Easy Ship,NaN,0.393732
4,4,407-1069790-7240320,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,INR,574.00,CHENNAI,Tamil Nadu,600073.0,IN,NaN,False,NaN,NaN,-0.263690


In [24]:
## check column names 
print(df.columns.tolist())

['idx', 'order_id', 'order_date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'currency', 'amount', 'ship_city', 'ship_state', 'ship_postal_code', 'ship_country', 'promotion_ids', 'b2b', 'fulfilled_by', 'unnamed_22', 'z_score']


In [26]:
## convert datatypes 
df["qty"] = pd.to_numeric(df["qty"], errors="coerce")
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df["status"] = df["status"].astype(str).str.strip()


In [27]:
df.shape

(128029, 25)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128030 entries, 0 to 128029
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   idx                 128030 non-null  object        
 1   order_id            128030 non-null  object        
 2   order_date          128029 non-null  datetime64[ns]
 3   status              128030 non-null  object        
 4   fulfilment          128030 non-null  object        
 5   sales_channel       128030 non-null  object        
 6   ship_service_level  128030 non-null  object        
 7   style               128030 non-null  object        
 8   sku                 128030 non-null  object        
 9   category            128030 non-null  object        
 10  size                128030 non-null  object        
 11  asin                128030 non-null  object        
 12  courier_status      121171 non-null  object        
 13  qty                 128029 no

## Return to sales Ratio

In [36]:
df["return_flag"] = np.where(
    df["status"] == "Cancelled",
    1,
    0
)

In [37]:
df[["status", "return_flag"]].head(10)

,status,return_flag
0,Cancelled,1
1,Shipped - Delivered to Buyer,0
2,Shipped,0
3,Cancelled,1
4,Shipped,0
5,Shipped,0
6,Shipped,0
7,Shipped - Delivered to Buyer,0
8,Cancelled,1
9,Shipped,0


In [38]:
rsr = (df["return_flag"].sum() / len(df)) * 100
print(f"Return-to-Sales Ratio: {rsr:.2f}%")

Return-to-Sales Ratio: 14.27%


## Fulfillment Efficiency Index

In [39]:
df["fulfillment_efficiency"] = np.where(
    df["status"].str.contains("Shipped", case=False, na=False),
    df["qty"],
    0
)

In [40]:
df["fulfillment_efficiency"] = pd.to_numeric(
    df["fulfillment_efficiency"],
    errors="coerce"
)

In [41]:
fei = (df["fulfillment_efficiency"].sum()/df["qty"].sum()) * 100
print("Fulfillment Efficiency Index :", round(fei,2),"%")

Fulfillment Efficiency Index : 94.34 %


In [42]:
## verify new features 
df[[
    "status",
    "qty",
    "return_flag",
    "fulfillment_efficiency"
]].head(15)

,status,qty,return_flag,fulfillment_efficiency
0,Cancelled,0,1,0
1,Shipped - Delivered to Buyer,1,0,1
2,Shipped,1,0,1
3,Cancelled,0,1,0
4,Shipped,1,0,1
5,Shipped,1,0,1
6,Shipped,1,0,1
7,Shipped - Delivered to Buyer,1,0,1
8,Cancelled,0,1,0
9,Shipped,1,0,1


In [43]:
df[[
    "amount",
    "qty",
    "return_flag",
    "fulfillment_efficiency"
]].describe()

,amount,qty,return_flag,fulfillment_efficiency
count,128029.000000,128029.000000,128029.000000,128029.000000
mean,638.738539,0.902592,0.142702,0.851518
std,259.376409,0.310110,0.349770,0.366589
min,0.000000,0.000000,0.000000,0.000000
25%,458.000000,1.000000,0.000000,1.000000
50%,605.000000,1.000000,0.000000,1.000000
75%,771.000000,1.000000,0.000000,1.000000
max,1463.000000,15.000000,1.000000,15.000000


In [44]:
## final dataset 
df.to_csv("../data/processed/amazon_sales_feature_engineered.csv",index=False)
print("Dataset exported successfully.")

Dataset exported successfully.
